# Political Embeddings Inference Pipeline

This notebook handles the full-dataset inference pipeline:
1. **Load Context**: Extract all videos with political saliency (`political_saliency > 0`) from `data/out/context.parquet` (~112,000 videos).
2. **Format Text Representations**: Assemble the multi-field text representation (`Channel`, `Title`, `Description`, `Tags`) matching the model's training format.
3. **Load Model**: Load the fine-tuned `political-embedder-small` model with Apple Silicon (`mps`) or GPU acceleration.
4. **Inference**: Generate dense normalized embeddings ($d=768$).
5. **Save to Parquet**: Save the embeddings alongside metadata directly to `data/out/political_embeddings.parquet`.

In [ ]:
import sys
from pathlib import Path

# Ensure src directory is in Python path
ROOT = Path("../")
sys.path.insert(0, str(ROOT / "src"))

import torch
import polars as pl
import numpy as np
from sentence_transformers import SentenceTransformer

from config import MODEL_DIR, OUT_DIR

# Select compute device
device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active device: {device}")

## 1. Load Context & Filter Political Videos
Filter for videos where `political_saliency > 0` and format the textual metadata representation.

In [ ]:
context_path = OUT_DIR / "context.parquet"
print(f"Reading context data from: {context_path}")
df_context = pl.read_parquet(context_path)

# Filter for political videos with valid titles
df_political = (
    df_context
    .filter((pl.col("political_saliency") > 0) & pl.col("title").is_not_null())
    .with_columns(
        assembled_text=pl.concat_str(
            [
                pl.lit("Channel: "), pl.col("channel").fill_null(""), pl.lit("\n"),
                pl.lit("Title: "), pl.col("title").fill_null(""), pl.lit("\n"),
                pl.lit("Description: "), pl.col("description").fill_null(""), pl.lit("\n"),
                pl.lit("Tags: "), pl.col("tags").fill_null("")
            ]
        )
    )
)

print(f"Total videos in context: {len(df_context):,}")
print(f"Political videos to embed: {len(df_political):,} ({len(df_political)/len(df_context):.1%})")
df_political.select(["id", "channel", "title", "slant", "political_saliency"]).head(5)

## 2. Load `political-embedder-small` Model
Find the latest training checkpoint in `models/political-embedder-small/`.

In [ ]:
model_dir = MODEL_DIR / "political-embedder-small"

# Identify latest checkpoint directory
checkpoints = sorted(
    [d for d in model_dir.glob("checkpoint-*") if d.is_dir()],
    key=lambda p: int(p.name.split("-")[1]) if p.name.split("-")[1].isdigit() else 0
)
model_path = checkpoints[-1] if checkpoints else model_dir
print(f"Loading checkpoint from: {model_path}")

model = SentenceTransformer(str(model_path), device=device)
print(f"Model successfully loaded on {device}. Embedding dimension: {model.get_sentence_embedding_dimension()}")

## 3. Generate Dense Embeddings
Compute normalized $d=768$ dense embeddings for all political videos in batches.

In [ ]:
texts = df_political["assembled_text"].to_list()
print(f"Generating embeddings for {len(texts):,} videos...")

embeddings = model.encode(
    texts,
    batch_size=128,
    show_progress_bar=True,
    normalize_embeddings=True
)
embeddings = embeddings.astype(np.float32)
print(f"Embeddings complete. Shape: {embeddings.shape}")

## 4. Save Embeddings to Parquet
Attach metadata (`id`, `channel`, `title`, `slant`, `political_saliency`) and write to `data/out/political_embeddings.parquet`.

In [ ]:
EMBEDDINGS_PARQUET = OUT_DIR / "political_embeddings.parquet"
print(f"Writing embeddings to: {EMBEDDINGS_PARQUET}")

df_embeddings = df_political.select([
    "id", "channel", "title", "slant", "political_saliency"
]).with_columns(
    embedding=pl.Series("embedding", embeddings.tolist())
)

df_embeddings.write_parquet(EMBEDDINGS_PARQUET)
print(f"Done! Successfully saved {len(df_embeddings):,} rows to {EMBEDDINGS_PARQUET.name}.")